# 08. 수작업 음향 특징 추출

저장된 10초 구간에서 MFCC와 변화량, 스펙트럼·에너지 특징을 계산한다. 시간별 값을 평균·표준편차로 요약한 266차원 벡터를 LR과 SVM 입력으로 사용한다. 입력은 24 kHz mono다.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

SEGMENT_PATH = PROJECT_ROOT / "data/metadata/segment_manifest_10s.csv"

FEATURE_DIR = PROJECT_ROOT / "data/processed/features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_PATH = FEATURE_DIR / "handcrafted_features_10s_checkpoint.pkl"
OUTPUT_PATH = FEATURE_DIR / "handcrafted_features_10s.csv"

SR = 24_000
SEGMENT_SEC = 10.0
TARGET_SAMPLES = int(SR * SEGMENT_SEC)

N_FFT = 1024
HOP_LENGTH = 240
N_MELS = 128
N_MFCC = 40
FMAX = 12_000

CHECKPOINT_EVERY = 250

print("SEGMENT_PATH    :", SEGMENT_PATH)
print("OUTPUT_PATH     :", OUTPUT_PATH)
print("CHECKPOINT_PATH :", CHECKPOINT_PATH)
print("Target samples  :", TARGET_SAMPLES)

SEGMENT_PATH    : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/segment_manifest_10s.csv
OUTPUT_PATH     : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/features/handcrafted_features_10s.csv
CHECKPOINT_PATH : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/features/handcrafted_features_10s_checkpoint.pkl
Target samples  : 240000


## 1. 라이브러리 확인

`librosa`를 이용해 waveform 로딩과 audio feature 추출을 수행한다.

In [2]:
# 라이브러리 확인
try:
    import librosa

    print("librosa version:", librosa.__version__)
except ImportError:
    raise ImportError(
        "librosa가 설치되어 있지 않습니다. "
        "이 노트북의 새 코드 셀에서 `%pip install librosa soundfile`을 실행한 뒤 커널을 다시 시작하세요."
    )

librosa version: 0.11.0


## 2. Segment Manifest 로드 및 기본 확인

In [3]:
segments = pd.read_csv(SEGMENT_PATH)

print("===== SEGMENT MANIFEST =====")
print("Rows             :", len(segments))
print("Unique segment_id:", segments["segment_id"].nunique())
print("Unique tracks    :", segments["track_sample_id"].nunique())
print("Unique groups    :", segments["original_audio"].nunique())

print("\nSplit:")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(segments["split"].value_counts())

print("\nLabel:")
print(segments["label"].value_counts())

display(segments.head())

===== SEGMENT MANIFEST =====
Rows             : 10077
Unique segment_id: 10077
Unique tracks    : 3458
Unique groups    : 296

Split:
split
train    6967
test     1572
val      1538
Name: count, dtype: int64

Label:
label
FAKE    9189
REAL     888
Name: count, dtype: int64


,segment_id,track_sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,...,planning_duration_sec,planned_segments,segment_index,segment_role,start_sec,end_sec,segment_duration_sec,available_audio_sec,pad_sec,requires_padding
0,seg_000000,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False
1,seg_000001,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,1,middle,10.0,20.0,10.0,10.000000,0.000000,False
2,seg_000002,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,...,30.0,3,2,end,20.0,30.0,10.0,9.988571,0.011429,True
3,seg_000003,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,...,30.0,3,0,start,0.0,10.0,10.0,10.000000,0.000000,False
4,seg_000004,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,...,30.0,3,1,middle,10.0,20.0,10.0,10.000000,0.000000,False


## 3. 10초 Waveform 로딩 함수

각 segment의 `start_sec`부터 필요한 오디오를 읽는다.

- 10초보다 길면 정확히 240,000 samples로 자름
- 부족하면 뒤쪽을 0으로 padding
- stereo는 mono로 변환
- 모든 파일은 24 kHz로 resampling

In [4]:
# 10초 Waveform 로딩 함수
def load_segment_waveform(row):
    full_path = PROJECT_ROOT / row["audio_path"]

    if not full_path.exists():
        raise FileNotFoundError(full_path)

    y, _ = librosa.load(
        full_path,
        sr=SR,
        mono=True,
        offset=float(row["start_sec"]),
        duration=SEGMENT_SEC,
    )

    y = np.asarray(y, dtype=np.float32)

    if len(y) < TARGET_SAMPLES:
        y = np.pad(y, (0, TARGET_SAMPLES - len(y)), mode="constant")
    elif len(y) > TARGET_SAMPLES:
        y = y[:TARGET_SAMPLES]

    if len(y) != TARGET_SAMPLES:
        raise RuntimeError(f"waveform length error: {len(y)} != {TARGET_SAMPLES}")

    if not np.isfinite(y).all():
        raise ValueError("waveform contains NaN or Inf")

    return y

## 4. Feature 추출 함수

각 feature는 시간축에 따라 여러 frame을 가지므로
최종적으로 각 feature 채널의 **mean / std**를 계산한다.

예를 들어 MFCC는 40차원이므로:

```text
mfcc_01_mean
mfcc_01_std
...
mfcc_40_mean
mfcc_40_std
```

형태가 된다.

Δ와 Δ²도 동일하게 계산한다.

In [5]:
def add_mean_std(feature_dict, prefix, x):
    x = np.asarray(x)

    if x.ndim == 1:
        x = x[np.newaxis, :]

    for i in range(x.shape[0]):
        name = f"{prefix}_{i+1:02d}" if x.shape[0] > 1 else prefix

        feature_dict[f"{name}_mean"] = float(np.mean(x[i]))
        feature_dict[f"{name}_std"] = float(np.std(x[i]))


def extract_handcrafted_features(y):
    features = {}

    # MFCC
    mfcc = librosa.feature.mfcc(
        y=y,
        sr=SR,
        n_mfcc=N_MFCC,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        n_mels=N_MELS,
        fmax=FMAX,
    )

    mfcc_delta = librosa.feature.delta(mfcc, order=1)
    mfcc_delta2 = librosa.feature.delta(mfcc, order=2)

    add_mean_std(features, "mfcc", mfcc)
    add_mean_std(features, "mfcc_delta", mfcc_delta)
    add_mean_std(features, "mfcc_delta2", mfcc_delta2)

    # Spectral features
    centroid = librosa.feature.spectral_centroid(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    bandwidth = librosa.feature.spectral_bandwidth(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    rolloff = librosa.feature.spectral_rolloff(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
        roll_percent=0.85,
    )

    flatness = librosa.feature.spectral_flatness(
        y=y,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    contrast = librosa.feature.spectral_contrast(
        y=y,
        sr=SR,
        n_fft=N_FFT,
        hop_length=HOP_LENGTH,
    )

    rms = librosa.feature.rms(
        y=y,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH,
    )

    zcr = librosa.feature.zero_crossing_rate(
        y=y,
        frame_length=N_FFT,
        hop_length=HOP_LENGTH,
    )

    add_mean_std(features, "spectral_centroid", centroid)
    add_mean_std(features, "spectral_bandwidth", bandwidth)
    add_mean_std(features, "spectral_rolloff", rolloff)
    add_mean_std(features, "spectral_flatness", flatness)
    add_mean_std(features, "spectral_contrast", contrast)
    add_mean_std(features, "rms", rms)
    add_mean_std(features, "zcr", zcr)

    return features

## 5. 한 Segment만 시험 추출

전체 10,077개를 처리하기 전에 첫 segment 하나가 정상적으로 로드되고
feature가 생성되는지 확인한다.

In [6]:
# 한 Segment만 시험 추출
test_row = segments.iloc[0]

test_y = load_segment_waveform(test_row)
test_features = extract_handcrafted_features(test_y)

print("Waveform shape :", test_y.shape)
print("Waveform dtype :", test_y.dtype)
print("Feature count  :", len(test_features))

print("\nFirst 10 features:")
for key in list(test_features.keys())[:10]:
    print(key, "=", test_features[key])

Waveform shape : (240000,)
Waveform dtype : float32
Feature count  : 266

First 10 features:
mfcc_01_mean = -148.81710815429688
mfcc_01_std = 31.026493072509766
mfcc_02_mean = 91.07467651367188
mfcc_02_std = 14.709043502807617
mfcc_03_mean = 19.076932907104492
mfcc_03_std = 8.961942672729492
mfcc_04_mean = 24.712404251098633
mfcc_04_std = 7.163117408752441
mfcc_05_mean = 8.618364334106445
mfcc_05_std = 6.378509044647217


### 기대 feature 차원

현재 설정에서 feature 수는 다음과 같다.

- MFCC: 40 × mean/std = 80
- Δ: 40 × mean/std = 80
- Δ²: 40 × mean/std = 80
- centroid: 2
- bandwidth: 2
- rolloff: 2
- flatness: 2
- contrast: 7 × mean/std = 14
- RMS: 2
- ZCR: 2

총 **266개 feature**가 기대된다.

In [7]:
EXPECTED_FEATURE_COUNT = 266

print("Expected features:", EXPECTED_FEATURE_COUNT)
print("Actual features  :", len(test_features))

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert (
    len(test_features) == EXPECTED_FEATURE_COUNT
), f"Feature count mismatch: {len(test_features)}"

Expected features: 266
Actual features  : 266


## 6. 전체 Segment Feature 추출

10,077개 segment를 순차적으로 처리한다.

시간이 오래 걸릴 수 있으므로:

- `CHECKPOINT_EVERY = 250` segment마다 checkpoint 저장
- 중간에 중단되어도 checkpoint가 있으면 이미 처리한 segment는 건너뜀
- 오류가 발생한 segment는 `error` 컬럼에 기록

In [8]:
# 전체 Segment Feature 추출
META_COLUMNS = [
    "segment_id",
    "track_sample_id",
    "original_audio",
    "label",
    "label_id",
    "source",
    "genre",
    "generator",
    "audio_path",
    "track_id",
    "split",
    "segment_index",
    "segment_role",
    "start_sec",
    "end_sec",
    "requires_padding",
    "pad_sec",
]

if CHECKPOINT_PATH.exists():
    feature_df = pd.read_pickle(CHECKPOINT_PATH)
    print("Loaded checkpoint rows:", len(feature_df))
else:
    feature_df = pd.DataFrame()
    print("No checkpoint found. Starting from zero.")

processed_ids = (
    set(feature_df["segment_id"].astype(str))
    if len(feature_df) and "segment_id" in feature_df.columns
    else set()
)

print("Already processed:", len(processed_ids))
print("Remaining        :", len(segments) - len(processed_ids))

No checkpoint found. Starting from zero.
Already processed: 0
Remaining        : 10077


In [9]:
# 전체 Segment Feature 추출
feature_rows = []

remaining = segments[~segments["segment_id"].astype(str).isin(processed_ids)]

for n, (_, row) in enumerate(remaining.iterrows(), start=1):
    result = {col: row[col] for col in META_COLUMNS if col in row.index}

    result["error"] = ""

    try:
        y = load_segment_waveform(row)
        feats = extract_handcrafted_features(y)

        result.update(feats)

    except Exception as e:
        result["error"] = f"{type(e).__name__}: {e}"

    feature_rows.append(result)

    if n % CHECKPOINT_EVERY == 0 or n == len(remaining):
        new_df = pd.DataFrame(feature_rows)

        if len(feature_df):
            combined = pd.concat(
                [feature_df, new_df],
                ignore_index=True,
            )
        else:
            combined = new_df.copy()

        combined = combined.drop_duplicates("segment_id", keep="last").reset_index(
            drop=True
        )

        combined.to_pickle(CHECKPOINT_PATH)

        print(
            f"processed this run: {n}/{len(remaining)} | "
            f"checkpoint rows: {len(combined)}"
        )

        feature_df = combined
        feature_rows = []

print("\nExtraction finished.")
print("Rows:", len(feature_df))

processed this run: 250/10077 | checkpoint rows: 250
processed this run: 500/10077 | checkpoint rows: 500
processed this run: 750/10077 | checkpoint rows: 750
processed this run: 1000/10077 | checkpoint rows: 1000
processed this run: 1250/10077 | checkpoint rows: 1250
processed this run: 1500/10077 | checkpoint rows: 1500
processed this run: 1750/10077 | checkpoint rows: 1750
processed this run: 2000/10077 | checkpoint rows: 2000
processed this run: 2250/10077 | checkpoint rows: 2250
processed this run: 2500/10077 | checkpoint rows: 2500
processed this run: 2750/10077 | checkpoint rows: 2750
processed this run: 3000/10077 | checkpoint rows: 3000
processed this run: 3250/10077 | checkpoint rows: 3250
processed this run: 3500/10077 | checkpoint rows: 3500
processed this run: 3750/10077 | checkpoint rows: 3750
processed this run: 4000/10077 | checkpoint rows: 4000
processed this run: 4250/10077 | checkpoint rows: 4250
processed this run: 4500/10077 | checkpoint rows: 4500
processed this r

## 7. 추출 오류 확인

In [10]:
# 추출 오류 확인
feature_df["error"] = feature_df["error"].fillna("")

failed = feature_df[feature_df["error"].str.len() > 0].copy()

print("Feature extraction success:", len(feature_df) - len(failed))
print("Feature extraction failed :", len(failed))

if len(failed):
    display(
        failed[
            [
                "segment_id",
                "track_sample_id",
                "label",
                "generator",
                "audio_path",
                "error",
            ]
        ].head(30)
    )

Feature extraction success: 10077
Feature extraction failed : 0


## 8. Feature 컬럼 및 결측값 검사

metadata를 제외한 숫자 feature만 선택하여
NaN / Inf가 존재하는지 확인한다.

In [11]:
FEATURE_COLUMNS = sorted(
    [
        c
        for c in feature_df.columns
        if (
            c.startswith("mfcc_")
            or c.startswith("mfcc_delta_")
            or c.startswith("mfcc_delta2_")
            or c.startswith("spectral_")
            or c.startswith("rms_")
            or c.startswith("zcr_")
        )
    ]
)

print("Feature columns:", len(FEATURE_COLUMNS))

numeric_features = feature_df[FEATURE_COLUMNS].apply(pd.to_numeric, errors="coerce")

nan_count = int(numeric_features.isna().sum().sum())
inf_count = int(np.isinf(numeric_features.to_numpy(dtype=float)).sum())

print("NaN values:", nan_count)
print("Inf values:", inf_count)

# ID와 행 수가 틀리면 특징·라벨 대응이 어긋나므로 여기서 확인한다.
assert len(FEATURE_COLUMNS) == EXPECTED_FEATURE_COUNT

Feature columns: 266
NaN values: 0
Inf values: 0


## 9. Segment Manifest와 정확히 대응하는지 검증

feature 결과의 segment ID가 원본 `segment_manifest_10s.csv`와
정확히 1:1로 일치하는지 확인한다.

In [12]:
# Segment Manifest와 정확히 대응하는지 검증
expected_ids = set(segments["segment_id"].astype(str))
feature_ids = set(feature_df["segment_id"].astype(str))

missing_ids = expected_ids - feature_ids
extra_ids = feature_ids - expected_ids

print("Expected segments:", len(expected_ids))
print("Feature rows     :", len(feature_ids))
print("Missing IDs      :", len(missing_ids))
print("Extra IDs        :", len(extra_ids))
print("Duplicate IDs    :", int(feature_df["segment_id"].duplicated().sum()))

Expected segments: 10077
Feature rows     : 10077
Missing IDs      : 0
Extra IDs        : 0
Duplicate IDs    : 0


## 10. Split / Label 분포 유지 확인

In [13]:
print("===== FEATURE ROWS BY SPLIT =====")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(feature_df["split"].value_counts())

print("\n===== FEATURE ROWS BY LABEL =====")
print(feature_df["label"].value_counts())

print("\n===== SPLIT × LABEL =====")
display(pd.crosstab(feature_df["split"], feature_df["label"]))

===== FEATURE ROWS BY SPLIT =====
split
train    6967
test     1572
val      1538
Name: count, dtype: int64

===== FEATURE ROWS BY LABEL =====
label
FAKE    9189
REAL     888
Name: count, dtype: int64

===== SPLIT × LABEL =====


label,FAKE,REAL
split,,
test,1437,135
train,6346,621
val,1406,132


## 11. 간단한 Feature 통계 확인

극단적인 NaN/Inf뿐 아니라 feature 값의 범위가 대략 정상인지 확인한다.

StandardScaler는 다음 모델 단계에서 **Train 데이터에만 fit**한다.
이 단계에서는 scaling하지 않는다.

In [14]:
# 간단한 Feature 통계 확인
feature_summary = numeric_features.describe().T

display(feature_summary.head(20))

print("\nFeature mean range:")
print(feature_summary["mean"].min(), "~", feature_summary["mean"].max())

print("\nFeature std range:")
print(feature_summary["std"].min(), "~", feature_summary["std"].max())

,count,mean,std,min,25%,50%,75%,max
mfcc_01_mean,10077.0,-294.112774,171.625996,-1130.038452,-378.068665,-261.542847,-173.596588,14.597221
mfcc_01_std,10077.0,83.412368,48.638673,5.074992,46.946857,72.820114,110.696510,300.103577
mfcc_02_mean,10077.0,104.400866,50.491222,-79.254478,70.361595,102.902122,137.320389,271.988953
mfcc_02_std,10077.0,34.780111,16.994651,2.818909,22.662245,32.040035,44.316521,121.074074
mfcc_03_mean,10077.0,-3.626517,34.776502,-146.416245,-25.017967,-1.623832,17.248669,123.405128
mfcc_03_std,10077.0,22.844807,10.651464,1.884892,15.110641,21.251238,28.754366,88.825279
mfcc_04_mean,10077.0,27.875391,22.041789,-92.356483,14.863726,26.819763,39.285301,140.488846
mfcc_04_std,10077.0,16.456048,6.904497,2.410260,11.404815,15.397865,20.381073,70.064766
mfcc_05_mean,10077.0,1.256213,16.636508,-122.661774,-7.141948,3.144531,11.635722,63.586517
mfcc_05_std,10077.0,13.441158,5.348704,1.977546,9.548738,12.562002,16.457150,54.154633



Feature mean range:
-294.1127740757988 ~ 4232.221282829258

Feature std range:
0.0038209879745913448 ~ 2111.104075697477


## 12. 최종 Feature QC

In [15]:
# 최종 Feature QC
qc_summary = pd.DataFrame(
    {
        "check": [
            "segment_manifest_rows",
            "feature_rows",
            "unique_segment_id",
            "duplicate_segment_id",
            "feature_columns",
            "extraction_failures",
            "missing_segment_ids",
            "extra_segment_ids",
            "nan_feature_values",
            "inf_feature_values",
            "missing_split",
            "missing_label",
        ],
        "value": [
            len(segments),
            len(feature_df),
            feature_df["segment_id"].nunique(),
            int(feature_df["segment_id"].duplicated().sum()),
            len(FEATURE_COLUMNS),
            len(failed),
            len(missing_ids),
            len(extra_ids),
            nan_count,
            inf_count,
            int(feature_df["split"].isna().sum()),
            int(feature_df["label"].isna().sum()),
        ],
    }
)

display(qc_summary)

core_qc_pass = (
    len(feature_df) == len(segments)
    and feature_df["segment_id"].nunique() == len(segments)
    and int(feature_df["segment_id"].duplicated().sum()) == 0
    and len(FEATURE_COLUMNS) == EXPECTED_FEATURE_COUNT
    and len(failed) == 0
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and nan_count == 0
    and inf_count == 0
    and int(feature_df["split"].isna().sum()) == 0
    and int(feature_df["label"].isna().sum()) == 0
)

print("===== FINAL RESULT =====")
print("Handcrafted Feature Core QC PASS:", core_qc_pass)

,check,value
0,segment_manifest_rows,10077
1,feature_rows,10077
2,unique_segment_id,10077
3,duplicate_segment_id,0
4,feature_columns,266
5,extraction_failures,0
6,missing_segment_ids,0
7,extra_segment_ids,0
8,nan_feature_values,0
9,inf_feature_values,0


===== FINAL RESULT =====
Handcrafted Feature Core QC PASS: True


## 13. 최종 Feature CSV 저장

QC 통과 후 최종 feature table을 CSV로 저장한다.

`handcrafted_features_10s.csv`는 다음 단계의
Logistic Regression / RBF-SVM에서 직접 사용한다.

In [16]:
if not core_qc_pass:
    raise RuntimeError(
        "Handcrafted feature QC가 통과하지 않았습니다. "
        "저장 전에 위 결과를 확인하세요."
    )

feature_df = feature_df.sort_values("segment_id").reset_index(drop=True)

feature_df.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", OUTPUT_PATH)
print("Rows :", len(feature_df))
print("Features:", len(FEATURE_COLUMNS))

Saved: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/processed/features/handcrafted_features_10s.csv
Rows : 10077
Features: 266


## 이어지는 기록

266차원 특징은 09번의 LR·SVM 입력으로 사용한다.

## 특징 추출 결과

- 10,077개 segment 모두에서 **266개 handcrafted feature**를 추출했다.
- 누락·추가·중복 segment ID와 NaN/Inf는 모두 0개다.
- Split은 Train 6,967, Validation 1,538, Test 1,572행으로 segment manifest와 정확히 일치한다.
- 결과를 `data/processed/features/handcrafted_features_10s.csv`에 저장했다.